# Fine-tune a poem model on Colab

LoRA-tunes a causal language model on the poem JSONL with [Unsloth](https://unsloth.ai/). Set `HF_MODEL` in the configuration cell. The notebook does not assume a particular model. Loss is on the poem tokens only.

Before running:

1. Upload this notebook to [Google Colab](https://colab.research.google.com/).
2. **Runtime → Change runtime type → A100 GPU**.
3. Set `HF_MODEL`. A few that fit this corpus:
   - `unsloth/Qwen2.5-3B-Instruct` (default, bf16 on a 40GB A100)
   - `unsloth/Qwen2.5-7B-Instruct` (lower `BATCH_SIZE` to 2 or 4 on a 40GB A100)
   - `sarvamai/sarvam-1` (completion model, no chat template; the notebook trains it as prompt and completion)
4. Run the cells from the top. The configuration cell mounts Google Drive and, the first time only, asks you to upload `data/sft/train.jsonl` (5.7 MB) and `data/sft/valid.jsonl` (0.6 MB). Those files stay in `My Drive/tagore-sft/`.
5. Short poems are packed into blocks of `MAX_LENGTH`, so the A100 is not waiting on padding. A checkpoint is copied to that Drive folder. If the session dies, rerun from the top with the same `HF_MODEL`, `LIMIT`, `EPOCHS`, `MAX_LENGTH`, `BATCH_SIZE`, and `LOAD_IN_4BIT`. It continues from the latest checkpoint.
6. When training finishes, the last cell downloads the adapter zip.

`LIMIT = 32` is a smoke test. `LIMIT = None` uses the full training file. One epoch is the first real run. Poems longer than `MAX_LENGTH` are skipped, so the model is not taught to stop mid-poem.


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
import json
import re
import shutil
from pathlib import Path

import torch
from google.colab import drive

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Use an A100 runtime: Runtime → Change runtime type → A100, then rerun from the top.")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("gpu", torch.cuda.get_device_name(0))

# Change this. Any causal LM Unsloth can load.
HF_MODEL = "unsloth/Qwen2.5-3B-Instruct"
# A100 trains 3B and 7B faster in bf16 than in 4-bit. Set True for a model that does not fit in 40GB.
LOAD_IN_4BIT = False
MAX_LENGTH = 2048
EPOCHS = 1
LIMIT = None  # 32 for a smoke test
# 3B bf16, packed to 2048, on a 40GB A100. Use 2 or 4 for a 7B model on 40GB.
BATCH_SIZE = 8
GRAD_ACCUM = 2  # effective batch 16
LORA_R = 16
LORA_ALPHA = 16
SAVE_STEPS = 10
SEED = 3407

drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/tagore-sft")
DRIVE_CKPT = DATA / "checkpoints"
model_slug = re.sub(r"[^A-Za-z0-9._-]+", "-", HF_MODEL.split("/")[-1])
ADAPTER_DIR = Path("/content") / f"{model_slug}-tagore-lora"

DATA.mkdir(parents=True, exist_ok=True)
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

needed = ["train.jsonl", "valid.jsonl"]
if not all((DATA / name).exists() for name in needed):
    from google.colab import files
    print("Upload train.jsonl and valid.jsonl from data/sft/. They are copied to Google Drive.")
    uploaded = files.upload()
    for name, blob in uploaded.items():
        (DATA / Path(name).name).write_bytes(blob)

missing = [name for name in needed if not (DATA / name).exists()]
if missing:
    raise RuntimeError(f"Missing {missing}. Upload both jsonl files.")


def latest_checkpoint(root):
    found = []
    for path in root.glob("checkpoint-*"):
        if not (path / "trainer_state.json").exists():
            continue
        step = path.name.split("-")[-1]
        if step.isdigit():
            found.append((int(step), path))
    return max(found)[1] if found else None


run_config = {
    "model": HF_MODEL,
    "load_in_4bit": LOAD_IN_4BIT,
    "limit": LIMIT,
    "epochs": EPOCHS,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "lora_r": LORA_R,
}
config_path = DRIVE_CKPT / "run_config.json"
saved = latest_checkpoint(DRIVE_CKPT)
if saved and config_path.exists():
    previous = json.loads(config_path.read_text(encoding="utf-8"))
    if previous != run_config:
        raise RuntimeError(
            f"Drive checkpoint used {previous}, this run uses {run_config}. "
            "Keep HF_MODEL, LOAD_IN_4BIT, LIMIT, EPOCHS, MAX_LENGTH, and BATCH_SIZE the same, "
            "or delete My Drive/tagore-sft/checkpoints to start over."
        )
else:
    config_path.write_text(json.dumps(run_config), encoding="utf-8")

RESUME = None
if saved:
    local = ADAPTER_DIR / saved.name
    if local.exists():
        shutil.rmtree(local)
    shutil.copytree(saved, local)
    RESUME = str(local)
    print("resuming from", saved)
else:
    print("no Drive checkpoint; training starts at step 0")
print("model", HF_MODEL)
print("data", DATA)


In [ ]:
def load_jsonl(path):
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def messages_of(row):
    return [item for item in row["messages"] if item["role"] in {"system", "user", "assistant"}]


train_rows = load_jsonl(DATA / "train.jsonl")
valid_rows = load_jsonl(DATA / "valid.jsonl")
if LIMIT:
    train_rows = train_rows[:LIMIT]
    valid_rows = valid_rows[: max(8, LIMIT // 5)]
print(f"train {len(train_rows)}  valid {len(valid_rows)}")
sample = messages_of(train_rows[0])
print(next(item["content"] for item in sample if item["role"] == "user"))
poem = next(item["content"] for item in sample if item["role"] == "assistant")
print("\n".join(poem.splitlines()[:4]))


## Train

Unsloth packs several short poems into each block of `MAX_LENGTH` (2048). On this corpus the median poem is a few hundred tokens, so packing is what keeps an A100 busy. A 4-bit load is slower here than bf16 when the weights fit, so `LOAD_IN_4BIT` stays off unless you pick a larger model.

Loss is on the assistant poem only. A model with a chat template is trained as a conversation. A completion model such as Sarvam-1 is trained as a prompt followed by the poem.

Every 10 steps the finished checkpoint is copied to `My Drive/tagore-sft/checkpoints`. A later run of this notebook loads that copy and continues.


In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer


class SyncCheckpoint(TrainerCallback):
    """Copy a finished checkpoint to Drive. A crash mid-copy leaves the previous one intact."""

    def on_save(self, args, state, control, **kwargs):
        src = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not (src / "trainer_state.json").exists():
            return
        dest = DRIVE_CKPT / src.name
        tmp = DRIVE_CKPT / f".{src.name}.partial"
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(src, tmp)
        if dest.exists():
            shutil.rmtree(dest)
        tmp.rename(dest)
        kept = sorted(
            (int(path.name.split("-")[-1]), path)
            for path in DRIVE_CKPT.glob("checkpoint-*")
            if path.name.split("-")[-1].isdigit()
        )
        for _, path in kept[:-2]:
            shutil.rmtree(path)
        print("checkpoint on Drive", dest)


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_MODEL,
    max_seq_length=MAX_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_16bit=not LOAD_IN_4BIT,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

wanted = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
present = {name.split(".")[-1] for name, _module in model.named_modules()}
target_modules = [name for name in wanted if name in present]
if not target_modules:
    raise RuntimeError(f"{HF_MODEL} has none of {wanted}.")
print("LoRA targets", target_modules)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    target_modules=target_modules,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_LENGTH,
    use_rslora=False,
)
model.print_trainable_parameters()

uses_chat = bool(getattr(tokenizer, "chat_template", None))
print("format", "chat" if uses_chat else "prompt-completion")


def prompt_and_poem(row):
    messages = messages_of(row)
    system = next((item["content"] for item in messages if item["role"] == "system"), "")
    user = next(item["content"] for item in messages if item["role"] == "user")
    poem = next(item["content"] for item in messages if item["role"] == "assistant")
    prompt = f"{system}\n\n{user.strip()}\n\n" if system else f"{user.strip()}\n\n"
    return prompt, poem


def overflows(row):
    if uses_chat:
        ids = tokenizer.apply_chat_template(
            messages_of(row), tokenize=True, add_generation_prompt=False
        )
        return len(ids) > MAX_LENGTH
    prompt, poem = prompt_and_poem(row)
    ids = tokenizer(prompt + poem, add_special_tokens=True)["input_ids"]
    return len(ids) + 1 > MAX_LENGTH


def build_dataset(rows):
    kept = [row for row in rows if not overflows(row)]
    if uses_chat:
        records = [{"messages": messages_of(row)} for row in kept]
    else:
        records = [
            {"prompt": prompt, "completion": poem}
            for prompt, poem in (prompt_and_poem(row) for row in kept)
        ]
    return Dataset.from_list(records), len(rows) - len(kept)


train_ds, train_skipped = build_dataset(train_rows)
valid_ds, valid_skipped = build_dataset(valid_rows)
print(f"train kept {len(train_ds)} skipped {train_skipped}")
print(f"valid kept {len(valid_ds)} skipped {valid_skipped}")
if len(train_ds) == 0:
    raise RuntimeError("Every training poem is longer than MAX_LENGTH.")

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    callbacks=[SyncCheckpoint()],
    args=SFTConfig(
        output_dir=str(ADAPTER_DIR),
        max_length=MAX_LENGTH,
        packing=True,
        packing_strategy="bfd",
        assistant_only_loss=uses_chat,
        dataset_num_proc=2,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="linear",
        weight_decay=0.01,
        optim="adamw_torch_fused",
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        bf16=True,
        fp16=False,
        seed=SEED,
        report_to="none",
    ),
)
trainer.train(resume_from_checkpoint=RESUME)
final = DATA / "adapter"
if final.exists():
    shutil.rmtree(final)
trainer.save_model(str(final))
tokenizer.save_pretrained(str(final))
print("adapter on Drive", final)


In [ ]:
THEMES = [
    "বিষয়: বৃষ্টি\nএই বিষয়ে একটি কবিতা লেখো।",
    "নদী নিয়ে একটি কবিতা লেখো।",
    "প্রসঙ্গ: সোনার তরী, 1894। ১৮৯৪। পদ্মার বোটে জমিদারি জীবন। বিষয়: সন্ধ্যা। একটি কবিতা লেখো।",
]

FastLanguageModel.for_inference(model)
system = next(item["content"] for item in messages_of(train_rows[0]) if item["role"] == "system")


def render_prompt(theme):
    if uses_chat:
        return tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": theme},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"{system}\n\n{theme.strip()}\n\n"


def adapter_generate(theme, max_new_tokens=512):
    prompt = render_prompt(theme)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


samples = []
for theme in THEMES:
    text = adapter_generate(theme)
    samples.append({"model": HF_MODEL, "prompt": theme, "completion": text})
    print("=" * 40)
    print(theme)
    print("-" * 40)
    print(text)
    print()

(DATA / "colab_samples.jsonl").write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in samples),
    encoding="utf-8",
)


In [ ]:
from google.colab import files

zip_source = DATA / "adapter" if (DATA / "adapter").exists() else ADAPTER_DIR
zip_path = shutil.make_archive(f"/content/{model_slug}-tagore-lora", "zip", zip_source)
print("download", zip_path)
files.download(zip_path)
